# Papers Topic Hierarchy (Low → Mid → High)

Builds a three-level hierarchy for the existing **papers** topic model
by cutting BERTopic's agglomerative merge tree at two levels:

- **mid** — target number of groups (configurable, default 40)
- **high** — best k in a configurable range, selected by silhouette score

Inputs:
- `assets/topic_models/papers_topic_model`
- `assets/topic_models/papers_doc_topics.txt`
- `assets/topic_models/papers_topic_names.txt`
- `assets/embeddings/papers_corpus.txt`
- `assets/synbio_openalex.txt`

Outputs:
- `assets/reports/papers_topic_hierarchy_map.tsv` — document-level mapping (ID, low, mid, high)
- `assets/reports/papers_topic_name_hierarchy.tsv` — topic names mapped to hierarchy (global_name, low, mid, high)
- `assets/reports/papers_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from bertopic import BERTopic

from hierarchy_utils import (
    build_hierarchy_maps,
    select_best_high_k,
    build_topic_hierarchy_df,
    build_doc_map,
    build_name_map,
    build_summary,
)

SEED = 42
np.random.seed(SEED)

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "topic_models"
EMBEDDINGS_DIR = ASSETS_DIR / "embeddings"
REPORTS_DIR = ASSETS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "papers_topic_model"
DOC_TOPICS_PATH = MODELS_DIR / "papers_doc_topics.txt"
TOPIC_NAMES_PATH = MODELS_DIR / "papers_topic_names.txt"
PAPERS_CORPUS_PATH = EMBEDDINGS_DIR / "papers_corpus.txt"
OPENALEX_PATH = ASSETS_DIR / "synbio_openalex.txt"

OUT_DOC_MAP = REPORTS_DIR / "papers_topic_hierarchy_map.tsv"
OUT_NAME_MAP = REPORTS_DIR / "papers_topic_name_hierarchy.tsv"
OUT_SUMMARY = REPORTS_DIR / "papers_topic_hierarchy_summary.tsv"

HIGH_K_MIN = 4
HIGH_K_MAX = 11
MID_K_TARGET = 40

In [ ]:
# Load model and input datasets
print("Loading model and datasets …")
papers_topic_model = BERTopic.load(str(MODEL_PATH))

papers_doc_topics = pd.read_csv(DOC_TOPICS_PATH, sep="\t")
papers_topic_names = pd.read_csv(TOPIC_NAMES_PATH, sep="\t")
papers_corpus = pd.read_csv(PAPERS_CORPUS_PATH, sep="\t")
papers_openalex = pd.read_csv(OPENALEX_PATH, sep="\t")

# Normalize key column names
papers_doc_topics = papers_doc_topics.rename(columns={"id": "ID", "topic": "low"})
papers_corpus = papers_corpus.rename(columns={"id": "ID"})
papers_openalex = papers_openalex.rename(columns={"id": "ID"})
papers_topic_names = papers_topic_names.rename(columns={"topic": "low"})

assert {"ID", "low"}.issubset(papers_doc_topics.columns)
assert {"low", "global_name"}.issubset(papers_topic_names.columns)
assert {"ID", "text"}.issubset(papers_corpus.columns)
assert {"ID", "publication_year"}.issubset(papers_openalex.columns)

papers_doc_topics["low"] = papers_doc_topics["low"].astype(int)
papers_topic_names["low"] = papers_topic_names["low"].astype(int)
papers_openalex["publication_year"] = pd.to_numeric(papers_openalex["publication_year"], errors="coerce")

print(f"  papers_doc_topics : {len(papers_doc_topics):,} rows")
print(f"  papers_topic_names: {len(papers_topic_names):,} rows")
print(f"  papers_corpus     : {len(papers_corpus):,} rows")
print(f"  papers_openalex   : {len(papers_openalex):,} rows")
print(f"  Outlier docs (low = -1): {(papers_doc_topics['low'] == -1).sum():,}")

papers_doc_topics: 24,202 rows
papers_topic_names: 263 rows
papers_corpus: 24,202 rows
papers_openalex: 24,202 rows
Outlier docs (low = -1): 9,017


In [ ]:
# Build hierarchy and select levels
print("Building hierarchy …")
corpus_texts = papers_corpus["text"].astype(str).tolist()

hier_df, maps_by_k, low_topics = build_hierarchy_maps(
    papers_topic_model, corpus_texts,
    max_k=max(HIGH_K_MAX, MID_K_TARGET),
)

# Ensure MID_K_TARGET exists in the map; fall back to nearest available
if MID_K_TARGET not in maps_by_k:
    MID_K_TARGET = max(k for k in maps_by_k if k >= HIGH_K_MAX)
    print(f"  Adjusted MID_K_TARGET → {MID_K_TARGET}")

selected_high_k, selected_high_score, candidate_scores = select_best_high_k(
    papers_topic_model, low_topics, maps_by_k,
    k_min=HIGH_K_MIN, k_max=HIGH_K_MAX,
)

low_to_mid = maps_by_k[MID_K_TARGET]
low_to_high = maps_by_k[selected_high_k]

topic_hierarchy_map = build_topic_hierarchy_df(low_topics, low_to_mid, low_to_high)

pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")

100%|██████████| 262/262 [00:10<00:00, 26.17it/s]


Low topics: 263
Mid groups (target): 40
High groups (selected): 11 | silhouette=0.1733


,high_k,silhouette
0,4,0.148989
1,5,0.146081
2,6,0.139180
3,7,0.148013
4,8,0.142613
5,9,0.156006
6,10,0.164337
7,11,0.173264


In [ ]:
# Report 1: paper-level mapping (ID, low, mid, high)
print("Building document map …")
report1 = build_doc_map(papers_doc_topics, topic_hierarchy_map, id_col="ID")
report1.to_csv(OUT_DOC_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_DOC_MAP.name}")
report1.head()

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_hierarchy_map.tsv


,ID,low,mid,high
0,https://openalex.org/W2072812562,167,24,5
1,https://openalex.org/W2091309425,167,24,5
2,https://openalex.org/W2037457710,44,16,5
3,https://openalex.org/W1984248485,7,6,0
4,https://openalex.org/W1980762198,65,27,3


In [ ]:
# Report 2: low-level global names mapped to low/mid/high
print("Building name map …")
report2 = build_name_map(papers_topic_names, topic_hierarchy_map)
report2.to_csv(OUT_NAME_MAP, sep="\t", index=False)

print(f"  Saved → {OUT_NAME_MAP.name}")
report2.head()

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_name_hierarchy.tsv


,global_name,low,mid,high
0,Genetic Circuit Control,0,0,0
1,Synthetic Nucleic Acid Design,1,1,1
2,Plant Biosynthetic Pathways,2,2,2
3,Synthetic CpG ODNs in Immunity,3,3,3
4,Ethics and Governance in Synthetic Biology,4,4,4


In [ ]:
# Report 3: mid/high summaries with count, average year, median year
print("Building summary …")
report3 = build_summary(
    report1, papers_openalex,
    id_col="ID", year_col="publication_year",
)
report3.to_csv(OUT_SUMMARY, sep="\t", index=False)

print(f"  Saved → {OUT_SUMMARY.name}")
report3.head(10)

Saved: /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/papers_topic_hierarchy_summary.tsv


,level,group_id,total_count,avg_publication_year,median_publication_year
0,high,0,2603,2016.815597,2018.0
1,high,1,1802,2003.721421,2005.0
2,high,2,2024,2017.669960,2020.0
3,high,3,1154,1998.142114,1997.0
4,high,4,1281,2016.663544,2017.0
5,high,5,2317,2010.947346,2016.0
6,high,6,1686,2015.708778,2019.0
7,high,7,508,2015.834646,2020.0
8,high,8,1128,2005.164007,2006.0
9,high,9,554,2007.516245,2011.0


In [ ]:
# Validation
assert list(report1.columns) == ["ID", "low", "mid", "high"]
assert list(report2.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(report3.columns)
assert len(report1) == len(papers_doc_topics)
assert (report1.loc[report1["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= selected_high_k <= HIGH_K_MAX

print("All validation checks passed ✓")
print(f"  report1: {len(report1):,} rows | report2: {len(report2):,} rows | report3: {len(report3):,} rows")

All validation checks passed.
Rows -> report1: 24,202, report2: 263, report3: 51


In [ ]:
# Hierarchy quality summary
quality = pd.DataFrame(candidate_scores, columns=["high_k", "silhouette"]).sort_values("high_k")
outlier_pct = (papers_doc_topics["low"] == -1).mean() * 100

print(f"Selected high-level K : {selected_high_k}")
print(f"Selected silhouette   : {selected_high_score:.4f}")
print(f"Outlier documents     : {outlier_pct:.2f}%")

quality

Selected high-level K: 11
Selected silhouette: 0.1733
Outlier documents (low = -1): 37.26%


,high_k,silhouette
0,4,0.148989
1,5,0.146081
2,6,0.139180
3,7,0.148013
4,8,0.142613
5,9,0.156006
6,10,0.164337
7,11,0.173264
